# Doctor-Patient Matching: Segmented ML Model

This notebook implements a machine learning pipeline to recommend doctors to patients based on cultural, linguistic, and specialty metrics. It uses a **segmented model strategy**, where separate 'expert' models are trained for different patient preference groups.

---

## Phase 1: Training the Expert Models

### Step 1: Load Data and Install Libraries

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import subprocess
import sys

# Install ethnicolr library for race prediction
try:
    from ethnicolr import census_ln
except ImportError:
    print("Installing ethnicolr...")
    # Note: ethnicolr requires TensorFlow. This installation might take a moment.
    subprocess.run([sys.executable, "-m", "pip", "install", "ethnicolr"], check=True)
    from ethnicolr import census_ln



In [3]:
# --- IMPORTANT: Update these file paths to match your system ---
#file locations
parquet_file_paths={
    "patient": r"Client_Data_files\Parquets\synthetic_patients.parquet",
    "encounter": r"Client_Data_files\Parquets\synthetic_encounters.parquet",
    "hospitals": r"Client_Data_files\Parquets\synthetic_hospitals.parquet",
    "provider": r"Client_Data_files\Parquets\synthetic_providers.parquet",    
}

# Reading the parquet files
patient_df = pd.read_parquet(parquet_file_paths['patient'])
encounter_df = pd.read_parquet(parquet_file_paths['encounter'])
hospital_df = pd.read_parquet(parquet_file_paths['hospitals'])
provider_df = pd.read_parquet(parquet_file_paths['provider'])

print("Dataframes loaded successfully.")
print(f"Patient DF shape: {patient_df.shape}")
print(f"Encounter DF shape: {encounter_df.shape}")
print(f"Provider DF shape: {provider_df.shape}")

Dataframes loaded successfully.
Patient DF shape: (100000, 16)
Encounter DF shape: (200000, 17)
Provider DF shape: (5000, 19)


In [5]:
# Creating a data map for easy access
data_map={
    "patient": patient_df,
    "encounter": encounter_df,
    "hospitals": hospital_df,
    "provider": provider_df
}

for key, df in data_map.items():
    print(f"Dataframe: {key}")
    print(f"Shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")    
    for col in df.columns:
        if df[col].isna().sum() > 0:
            print(f"Column '{col}' has {df[col].isna().sum()} missing values.")
    print("")

Dataframe: patient
Shape: (100000, 16)
Columns: ['patient_id', 'first_name', 'last_name', 'date_of_birth', 'gender', 'race', 'ethnicity', 'primary_language', 'zip_code', 'insurance_type', 'household_income', 'education_level', 'age', 'cultural_background', 'preferred_provider_language', 'cultural_preferences']

Dataframe: encounter
Shape: (200000, 17)
Columns: ['encounter_id', 'patient_id', 'provider_id', 'encounter_date', 'encounter_type', 'primary_diagnosis', 'length_of_stay', 'total_cost', 'cultural_background', 'primary_language', 'languages_spoken', 'cultural_competency_rating', 'cultural_match_score', 'language_match', 'patient_satisfaction', 'treatment_adherence', 'return_visit_30_days']

Dataframe: hospitals
Shape: (200, 14)
Columns: ['hospital_id', 'hospital_name', 'hospital_type', 'zip_code', 'bed_count', 'teaching_hospital', 'trauma_center', 'language_services_available', 'cultural_competency_program', 'interpreter_services_24_7', 'community_health_programs', 'overall_rating

### Step 2: Feature Engineering

In [6]:
print(patient_df['race'].value_counts())
print("")
print(patient_df['ethnicity'].value_counts())

race
White                        65181
Hispanic or Latino           12997
Black or African American    12011
Asian                         5932
Other                         1943
Native American               1936
Name: count, dtype: int64

ethnicity
Not Hispanic or Latino    84859
Hispanic or Latino        15141
Name: count, dtype: int64


In [7]:
patient_df_temp=patient_df[patient_df['race']=='Hispanic or Latino'][['patient_id','first_name','last_name','race','ethnicity']].copy()
patient_df_temp.head()

,patient_id,first_name,last_name,race,ethnicity
6,PAT_000007,Ming,Smith,Hispanic or Latino,Not Hispanic or Latino
15,PAT_000016,Li,Miller,Hispanic or Latino,Hispanic or Latino
20,PAT_000021,Fatima,Brown,Hispanic or Latino,Hispanic or Latino
29,PAT_000030,Ahmed,Wang,Hispanic or Latino,Not Hispanic or Latino
31,PAT_000032,William,Thomas,Hispanic or Latino,Not Hispanic or Latino


In [8]:
patient_race_pred=census_ln(patient_df_temp, 'last_name')
patient_race_pred.head()

2025-09-27 21:01:38,939 - INFO - Preserving 12965 duplicate rows based on column 'last_name'
2025-09-27 21:01:38,949 - INFO - Data filtering summary: 12997 → 12997 rows (kept 100.0%)
2025-09-27 21:01:38,996 - INFO - Loading Census 2000 data from c:\Users\jerry\anaconda3\envs\Env2709_Capstone_py_311\Lib\site-packages\ethnicolr\data\census\census_2000.csv...
2025-09-27 21:01:39,462 - INFO - Loaded 151670 last names from Census 2000
2025-09-27 21:01:39,471 - INFO - Merging demographic data for 12997 records...
2025-09-27 21:01:39,669 - INFO - Matched 12997 of 12997 rows (100.0%)
2025-09-27 21:01:39,670 - INFO - Added columns: pct2prace, pctaian, pctapi, pctblack, pcthispanic, pctwhite


,patient_id,first_name,last_name,race,ethnicity,pctwhite,pctblack,pctapi,pctaian,pct2prace,pcthispanic
0,PAT_000007,Ming,Smith,Hispanic or Latino,Not Hispanic or Latino,73.35,22.22,0.40,0.85,1.63,1.56
1,PAT_000016,Li,Miller,Hispanic or Latino,Hispanic or Latino,85.81,10.41,0.42,0.63,1.31,1.43
2,PAT_000021,Fatima,Brown,Hispanic or Latino,Hispanic or Latino,60.71,34.54,0.41,0.83,1.86,1.64
3,PAT_000030,Ahmed,Wang,Hispanic or Latino,Not Hispanic or Latino,3.25,0.19,94.47,0.03,1.73,0.33
4,PAT_000032,William,Thomas,Hispanic or Latino,Not Hispanic or Latino,55.53,38.17,1.63,1.01,2.00,1.66


In [9]:
race_mapping={
    'white': 'White',
    'black': 'Black or African American',
    'api': 'Asian',    
    'aian': 'Native American',
    '2prace': 'Other'
}

In [10]:
race_cols=['pctwhite','pctblack','pctapi','pctaian','pct2prace']
patient_race_pred['derived_race'] = patient_race_pred[race_cols].idxmax(axis=1).str.replace('pct', '').map(race_mapping)
patient_race_pred.head()

,patient_id,first_name,last_name,race,ethnicity,pctwhite,pctblack,pctapi,pctaian,pct2prace,pcthispanic,derived_race
0,PAT_000007,Ming,Smith,Hispanic or Latino,Not Hispanic or Latino,73.35,22.22,0.40,0.85,1.63,1.56,White
1,PAT_000016,Li,Miller,Hispanic or Latino,Hispanic or Latino,85.81,10.41,0.42,0.63,1.31,1.43,White
2,PAT_000021,Fatima,Brown,Hispanic or Latino,Hispanic or Latino,60.71,34.54,0.41,0.83,1.86,1.64,White
3,PAT_000030,Ahmed,Wang,Hispanic or Latino,Not Hispanic or Latino,3.25,0.19,94.47,0.03,1.73,0.33,Asian
4,PAT_000032,William,Thomas,Hispanic or Latino,Not Hispanic or Latino,55.53,38.17,1.63,1.01,2.00,1.66,White


In [11]:
patient_df[patient_df['race']=='Hispanic or Latino'].shape

(12997, 16)

In [12]:
patient_df[patient_df['race']=='Hispanic or Latino'].head()

,patient_id,first_name,last_name,date_of_birth,gender,race,ethnicity,primary_language,zip_code,insurance_type,household_income,education_level,age,cultural_background,preferred_provider_language,cultural_preferences
6,PAT_000007,Ming,Smith,1975-08-13 19:28:50.538981,F,Hispanic or Latino,Not Hispanic or Latino,Spanish,18210,Medicare,50335,High School,50,Other/Mixed,Spanish,Culturally Similar Provider; Same Language Pro...
15,PAT_000016,Li,Miller,1944-09-03 19:28:50.538993,M,Hispanic or Latino,Hispanic or Latino,Vietnamese,75629,Private,46488,Graduate,81,Hispanic/Latino,Vietnamese,Culturally Similar Provider; Same Language Pro...
20,PAT_000021,Fatima,Brown,1994-10-14 19:28:50.539000,M,Hispanic or Latino,Hispanic or Latino,English,25917,Medicare,105776,Bachelor's,30,Hispanic/Latino,English,Culturally Similar Provider
29,PAT_000030,Ahmed,Wang,1938-12-27 19:28:50.539012,M,Hispanic or Latino,Not Hispanic or Latino,English,69620,Uninsured,18641,Some College,86,Other/Mixed,English,Culturally Similar Provider
31,PAT_000032,William,Thomas,1990-09-02 19:28:50.539015,F,Hispanic or Latino,Not Hispanic or Latino,Spanish,35624,Medicare,44268,Bachelor's,35,Other/Mixed,Spanish,Culturally Similar Provider; Same Language Pro...


In [13]:
# Create a mapping from patient_id to derived_race
id_to_derived_race = dict(zip(patient_race_pred['patient_id'], patient_race_pred['derived_race']))


# Update the race column only for Hispanic or Latino patients
patient_df.loc[patient_df['race'] == 'Hispanic or Latino', 'race'] = \
    patient_df.loc[patient_df['race'] == 'Hispanic or Latino', 'patient_id'].map(id_to_derived_race)

In [14]:
provider_race_predictions = census_ln(provider_df, 'last_name')

2025-09-27 21:01:55,491 - INFO - Preserving 4968 duplicate rows based on column 'last_name'
2025-09-27 21:01:55,493 - INFO - Data filtering summary: 5000 → 5000 rows (kept 100.0%)
2025-09-27 21:01:55,507 - INFO - Merging demographic data for 5000 records...
2025-09-27 21:01:55,696 - INFO - Matched 5000 of 5000 rows (100.0%)
2025-09-27 21:01:55,699 - INFO - Added columns: pct2prace, pctaian, pctapi, pctblack, pcthispanic, pctwhite


In [15]:
# Derive race for the provider_df as it is missing from the source data
print("Deriving race for providers from last names...")
race_cols = ['pctwhite','pctblack','pctapi','pctaian','pct2prace']
provider_df['provider_race'] = provider_race_predictions[race_cols].idxmax(axis=1).str.replace('pct', '').map(race_mapping)
print("Provider race derivation complete.")

Deriving race for providers from last names...
Provider race derivation complete.


In [17]:
# Deriving provider ethnicity from the race_predictions
print("Deriving ethnicity for providers from race predictions...")
provider_df['provider_ethnicity'] = provider_race_predictions['pcthispanic'].apply(lambda x: 'Hispanic or Latino' if float(x) >= 50 else 'Not Hispanic or Latino')
print("Provider ethnicity derivation complete.")


Deriving ethnicity for providers from race predictions...
Provider ethnicity derivation complete.


In [18]:
# # Derive race for the provider_df as it is missing from the source data
# print("Deriving race for providers from last names...")
# # Note: This assumes a 'last_name' column exists in your provider_df
# race_predictions = census_ln(provider_df, 'last_name')
# race_cols = ['race_white', 'race_black', 'race_api', 'race_aian', 'race_2prace']
# provider_df['derived_race'] = race_predictions[race_cols].idxmax(axis=1).str.replace('race_', '')
# print("Provider race derivation complete.")

# Merge all data into a single master DataFrame for training
master_df = pd.merge(encounter_df, patient_df, on='patient_id',suffixes=('', '_pat'))
master_df = pd.merge(master_df, provider_df, on='provider_id',suffixes=('', '_prov'))
master_df = pd.merge(master_df, hospital_df, left_on='hospital_affiliation', right_on='hospital_id',suffixes=('', '_hosp'))

master_df.columns

Index(['encounter_id', 'patient_id', 'provider_id', 'encounter_date',
       'encounter_type', 'primary_diagnosis', 'length_of_stay', 'total_cost',
       'cultural_background', 'primary_language', 'languages_spoken',
       'cultural_competency_rating', 'cultural_match_score', 'language_match',
       'patient_satisfaction', 'treatment_adherence', 'return_visit_30_days',
       'first_name', 'last_name', 'date_of_birth', 'gender', 'race',
       'ethnicity', 'primary_language_pat', 'zip_code', 'insurance_type',
       'household_income', 'education_level', 'age', 'cultural_background_pat',
       'preferred_provider_language', 'cultural_preferences', 'npi_number',
       'first_name_prov', 'last_name_prov', 'specialty', 'practice_zip_code',
       'years_experience', 'medical_school_country', 'board_certified',
       'languages_spoken_prov', 'interpreter_services',
       'cultural_certifications', 'minority_health_experience',
       'community_involvement', 'patient_satisfaction_sc

In [21]:
# Engineer the match features
# Note: Ensure these column names ('primary_language', 'languages_spoken', etc.) match your dataframes
master_df['language_match_val'] = (master_df['language_match'] == 'TRUE').astype(int)
master_df['race_match_val'] = (master_df['race'] == master_df['provider_race']).astype(int)

print("Feature engineering complete. Master DataFrame created.")
master_df[['patient_id', 'provider_id', 'language_match_val', 'race_match_val', 'cultural_preferences']].head()

Feature engineering complete. Master DataFrame created.


,patient_id,provider_id,language_match_val,race_match_val,cultural_preferences
0,PAT_052723,PROV_04581,0,1,No Specific Preference
1,PAT_046599,PROV_00846,0,0,Culturally Similar Provider
2,PAT_016355,PROV_01047,0,0,No Specific Preference
3,PAT_016754,PROV_04343,0,1,No Specific Preference
4,PAT_074168,PROV_01228,0,1,No Specific Preference


### Step 3: Segment the Data

In [22]:
# Verify the distribution of patient preferences
print("Distribution of Cultural Preferences:")
print(master_df['cultural_preferences'].value_counts(normalize=True))

# Create the 'High-Preference' group by combining two categories
high_pref_categories = ['Same Language Provider', 'Culturally Similar Provider; Same Language Provider']
df_high_preference = master_df[master_df['cultural_preferences'].isin(high_pref_categories)].copy()

# Create the other two segments
df_cultural = master_df[master_df['cultural_preferences'] == 'Culturally Similar Provider'].copy()
df_pragmatist = master_df[master_df['cultural_preferences'] == 'No Specific Preference'].copy()

# A dictionary to hold our segmented data for easier processing
segmented_data = {
    "high_preference": df_high_preference,
    "cultural": df_cultural,
    "pragmatist": df_pragmatist
}

print("\nData segmentation complete:")
for name, df in segmented_data.items():
    print(f"- Segment '{name}' has {len(df)} encounters.")

Distribution of Cultural Preferences:
cultural_preferences
No Specific Preference                                 0.432195
Culturally Similar Provider                            0.346285
Same Language Provider                                 0.123620
Culturally Similar Provider; Same Language Provider    0.097900
Name: proportion, dtype: float64

Data segmentation complete:
- Segment 'high_preference' has 44304 encounters.
- Segment 'cultural' has 69257 encounters.
- Segment 'pragmatist' has 86439 encounters.


### Step 4: Train a Model for Each Segment

In [28]:
# Define the features the models will use
features = ['cultural_competency_rating', 'years_experience', 'language_match_val', 'race_match_val']
target = 'patient_satisfaction'

trained_models = {}

print("--- Starting Model Training Phase ---")
for segment_name, segment_df in segmented_data.items():
    print(f"\n--- Training model for segment: '{segment_name}' ---")
    
    # Set a reasonable minimum threshold for training
    if len(segment_df) < 50:
        print(f"Segment '{segment_name}' is too small to train a model. Skipping.")
        continue

    X = segment_df[features]
    y = segment_df[target]

    # Split data for validation (80% train, 20% test)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Handle cases where the split results in an empty test set for small segments
    if len(X_test) == 0:
        print(f"Not enough data in segment '{segment_name}' for a test split. Training on all data.")
        X_train, y_train = X, y # Train on all available data

    # Initialize and train the Random Forest Regressor model
    model = RandomForestRegressor(n_estimators=100, random_state=42, oob_score=True, n_jobs=-1)
    model.fit(X_train, y_train)

    # --- Testing Section ---
    if len(X_test) > 0:
        predictions = model.predict(X_test)
        # rmse = mean_squared_error(y_test, predictions, squared=False)
        rmse = mean_squared_error(y_test, predictions, )
        mae = mean_absolute_error(y_test, predictions)
        r2 = r2_score(y_test, predictions)
        print("Offline Validation Metrics:")
        print(f"  - RMSE: {rmse:.4f}")
        print(f"  - MAE:  {mae:.4f}")
        print(f"  - R²:   {r2:.4f}")

    # Store the fully trained model for inference
    trained_models[segment_name] = model
    
    # Display learned feature importances for this segment's model
    importances = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)
    print("\nLearned Feature Importances:")
    print(importances)

print("\n--- Model Training Complete ---")

--- Starting Model Training Phase ---

--- Training model for segment: 'high_preference' ---
Offline Validation Metrics:
  - RMSE: 0.1128
  - MAE:  0.2809
  - R²:   -0.0655

Learned Feature Importances:
cultural_competency_rating    0.473407
years_experience              0.412706
race_match_val                0.113887
language_match_val            0.000000
dtype: float64

--- Training model for segment: 'cultural' ---
Offline Validation Metrics:
  - RMSE: 0.0589
  - MAE:  0.2108
  - R²:   -0.0986

Learned Feature Importances:
cultural_competency_rating    0.492081
years_experience              0.390945
race_match_val                0.116974
language_match_val            0.000000
dtype: float64

--- Training model for segment: 'pragmatist' ---
Offline Validation Metrics:
  - RMSE: 0.0573
  - MAE:  0.2103
  - R²:   -0.0533

Learned Feature Importances:
cultural_competency_rating    0.541193
years_experience              0.372420
race_match_val                0.086386
language_match_val  

---

## Phase 2: Inference (Getting Recommendations)

In [33]:
def get_recommendations(patient_id, required_specialty, all_providers_df, all_patients_df, models_dict):
    """
    Generates ranked doctor recommendations using the segmented model strategy.
    """
    print(f"\n--- Starting Recommendation Phase for Patient ID: {patient_id} ---")
    
    # --- Step 1: Patient Lookup & Model Routing ---
    patient_info = all_patients_df[all_patients_df['patient_id'] == patient_id]
    if patient_info.empty:
        return "Error: Patient not found."
        
    preference = patient_info['cultural_preferences'].iloc[0]
    model_to_use = None
    model_name = ""
    
    if preference in high_pref_categories:
        model_name = 'high_preference'
    elif preference == 'Culturally Similar Provider':
        model_name = 'cultural'
    else:
        model_name = 'pragmatist'
        
    model_to_use = models_dict.get(model_name)
    print(f"Patient preference: '{preference}'. Routing to '{model_name}' model.")
        
    if not model_to_use:
        return f"Error: Model for preference group '{model_name}' was not trained (likely due to small size)."

    # --- Step 2: Candidate Generation ---
    candidate_providers = all_providers_df[all_providers_df['specialty'] == required_specialty].copy()
    if candidate_providers.empty:
        return f"Error: No providers found for the required specialty '{required_specialty}'."

    # --- Step 3: Create Feature Vectors ---
    # Create a DataFrame with a row for the patient paired with each candidate provider
    inference_df = candidate_providers.assign(key=1).merge(patient_info.assign(key=1), on='key').drop('key', axis=1)
    
    # Engineer the same features used in training
    inference_df['language_match'] = [1 if p_lang in d_langs else 0 for p_lang, d_langs in zip(inference_df['primary_language'], inference_df['languages_spoken'])]
    inference_df['race_match'] = (inference_df['race'] == inference_df['provider_race']).astype(int)
    
    # Ensure the columns are in the same order as during training
    X_inference = inference_df[features]

    # --- Step 4: Predict Scores ---
    predicted_scores = model_to_use.predict(X_inference)
    candidate_providers['predicted_satisfaction'] = predicted_scores

    # --- Step 5: Rank and Return ---
    recommendations = candidate_providers.sort_values(by='predicted_satisfaction', ascending=False)
    
    print("--- Recommendations Generated ---")
    return recommendations[['provider_id', 'specialty', 'predicted_satisfaction', 'derived_race', 'languages_spoken']]

### Example Usage

Now you can test the `get_recommendations` function with a real patient ID and specialty from your dataset.

In [34]:
# --- Replace these values with a real patient_id and specialty from your data ---
example_patient_id = 'PAT_074168' # Replace with a valid ID from patient_df
example_specialty = 'Cardiology' # Replace with a valid specialty from provider_df

# Ensure that there are models trained before running this
if trained_models:
    recommendations = get_recommendations(
        patient_id=example_patient_id,
        required_specialty=example_specialty,
        all_providers_df=provider_df,
        all_patients_df=patient_df,
        models_dict=trained_models
    )
    display(recommendations)
else:
    print("No models were trained, cannot generate recommendations. Please check the training phase.")


--- Starting Recommendation Phase for Patient ID: PAT_074168 ---
Patient preference: 'No Specific Preference'. Routing to 'pragmatist' model.


KeyError: "['language_match_val', 'race_match_val'] not in index"